# Pipeline de Transformação — Dengue, Zika e Chikungunya

Este notebook tem como objetivo **testar passo a passo** o pipeline de transformação
dos dados epidemiológicos, antes da criação do orquestrador final.

Pipeline testado:

1. Ajuste de path e importação dos módulos
2. Leitura e junção dos dados brutos por agravo e ano
3. Seleção e padronização de colunas
4. Padronização de tipos
5. Normalização de chaves territoriais e agravos
6. Validação de schema final

Ao final, teremos um DataFrame pronto para análises e dashboards.


## 1. Setup do Ambiente

Nesta etapa:
- Definimos caminhos base do projeto


In [1]:
from pathlib import Path
import sys

current = Path().resolve()

PROJECT_ROOT = None
for parent in [current] + list(current.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Não foi possível encontrar a pasta src")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: C:\Users\gfasg\OneDrive\Área de Trabalho\Projeto Dengue\dengue-territorio-e-condicoes-urbanas-recife


## 1. Importação dos módulos do pipeline

Aqui importamos todos os módulos de transformação que serão usados
pelo orquestrador no futuro.


In [2]:
from src.transform.join_tables import join_agravo_years
from src.transform.select_columns import (
    tratar_dados_dengue,
    tratar_dados_zika,
    tratar_dados_chikungunya
)
from src.transform.standardize_types import standardize_types
from src.transform.normalize_keys import normalize_keys
from src.transform.validate_schema import validate_schema


## 2. Leitura e junção dos dados brutos

Nesta etapa, consolidamos todos os anos disponíveis de cada agravo
em um único DataFrame por doença.


In [3]:
df_dengue_raw = join_agravo_years("dengue")
df_zika_raw = join_agravo_years("zika")
df_chik_raw = join_agravo_years("chikungunya")

df_dengue_raw.shape, df_zika_raw.shape, df_chik_raw.shape


((31325, 328), (1807, 94), (26613, 273))

## 3. Seleção e padronização de colunas por agravo

Cada função:
- Remove colunas desnecessárias
- Padroniza os nomes das colunas
- Mantém apenas o subconjunto definido no escopo do projeto


In [4]:
df_dengue = tratar_dados_dengue(df_dengue_raw)
df_zika = tratar_dados_zika(df_zika_raw)
df_chik = tratar_dados_chikungunya(df_chik_raw)

df_dengue.shape, df_zika.shape, df_chik.shape


((31325, 22), (1807, 11), (26613, 21))

## 4. Concatenação dos agravos

Após a padronização estrutural, os três agravos podem ser combinados
em um único DataFrame comparável.


In [5]:
import pandas as pd

df_all = pd.concat(
    [df_dengue, df_zika, df_chik],
    ignore_index=True
)

df_all.shape


(59745, 22)

## 5. Padronização de tipos

Conversão de:
- Datas
- Campos booleanos
- Códigos categóricos
- Inteiros padronizados

Esta etapa garante consistência para análises temporais e agregações.


In [6]:
df_all = standardize_types(df_all)

df_all.dtypes


num_notificacao                     object
dt_notificacao              datetime64[ns]
co_municipio_notificacao           float64
co_unidade_notificacao             float64
dt_nascimento               datetime64[ns]
sexo                                object
gestante                           float64
raca_cor                           float64
bairro_residencia                   object
febre                                Int64
mialgia                              Int64
cefaleia                             Int64
exantema                             Int64
vomito                               Int64
nausea                               Int64
dor_costas                           Int64
conjuntivite                         Int64
artrite                              Int64
artralgia                            Int64
classificacao_final                float64
ocorreu_hospitalizacao             float64
evolucao_caso                      float64
idade                              float64
dtype: obje

## 6. Normalização de chaves territoriais e agravos

Nesta etapa:
- UF, município, distrito e bairro são normalizados
- Join com tabelas auxiliares
- Nenhum município é descartado


In [7]:
df_all = normalize_keys(df_all)
df_all.head()


,num_notificacao,dt_notificacao,co_municipio_notificacao,co_unidade_notificacao,dt_nascimento,sexo,gestante,raca_cor,bairro_residencia,febre,...,vomito,nausea,dor_costas,conjuntivite,artrite,artralgia,classificacao_final,ocorreu_hospitalizacao,evolucao_caso,idade
0,NaN,2021-01-01,261160.0,671.0,1984-07-26,F,9.0,9.0,IBURA,1,...,0,0,0,0,0,0,10.0,NaN,1.0,36.0
1,NaN,2021-01-01,261160.0,671.0,1972-08-22,F,9.0,9.0,IBURA,1,...,0,0,0,0,0,0,10.0,NaN,1.0,48.0
2,NaN,2021-01-01,261160.0,671.0,1961-05-12,F,9.0,9.0,IBURA,1,...,0,0,0,0,0,0,10.0,NaN,1.0,59.0
3,NaN,2021-01-01,261160.0,671.0,1950-04-20,F,9.0,9.0,IBURA,1,...,0,0,0,0,0,0,10.0,NaN,1.0,70.0
4,NaN,2021-01-01,261160.0,590.0,1990-06-04,F,5.0,1.0,ESTANCIA,1,...,0,0,1,0,0,0,5.0,NaN,1.0,30.0


## 7. Validação de schema final

Checagens realizadas:
- Colunas obrigatórias
- Tipos esperados
- Campos críticos nulos

In [8]:
validate_schema(df_all)

df_all.shape


❌ Colunas obrigatórias ausentes:
  - agravo
  - ano_origem
  - tp_sexo
  - tp_gestante
  - tp_raca_cor
  - tp_zona_residencia
  - co_municipio_residencia
  - nome_municipio
  - co_bairro_residencia
  - bairro_padronizado
  - tp_classificacao_final
  - st_ocorreu_hospitalizacao
  - tp_evolucao_caso

⚠️ Problemas de tipo:
  - num_notificacao: esperado inteiro

📊 Percentual de valores nulos (top 10):
ocorreu_hospitalizacao    99.67
num_notificacao           87.20
idade                     82.48
dt_notificacao            79.20
dt_nascimento             43.34
febre                     37.13
mialgia                   37.13
artralgia                 37.13
artrite                   37.13
dor_costas                37.13
dtype: float64

⚠️ Schema inválido (modo permissivo)


(59745, 23)

## Conclusão

O pipeline de transformação foi executado com sucesso.

Próximos passos:
1. Criar o orquestrador do pipeline (src/transform/pipeline.py)
2. Criar o build_final_table
3. Integrar com os notebooks de visualização
